# Barcode Decode: Mono Camera (1px shift, downsampled 2x)

Run `sweep_sr_barcodes_mono_1px.py` first to generate SR outputs.

Mono images are spatially downsampled 2x to simulate Bayer red-channel extraction.
1px sensor shift = 0.5 LR pixel shift. Upsample factor 2 = back to full sensor resolution.

**Kernel: Python 3.10 (sr310)** (`conda activate sr310`)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict
import zxingcpp

NB_DIR        = os.path.dirname(os.path.abspath("barcode_decode_mono_1px.ipynb"))
METHOD_NAMES  = ["Native-2x", "SAA", "SAA+IBP"]
COLORS_METHOD = {"Native-2x": "C0", "SAA": "C2", "SAA+IBP": "C3"}
name_map      = {"Native-2x": "native_2x", "SAA": "SAA", "SAA+IBP": "SAA_IBP"}

# ── Discover sessions ──────────────────────────────────────────────────────
sessions_available = sorted([
    d for d in os.listdir(NB_DIR)
    if os.path.isdir(os.path.join(NB_DIR, d))
    and d.startswith('MONO_BARCODES_1PIXELSHIFT')
])
print("Sessions:")
for s in sessions_available:
    print(f"  {s}")

# ── Discover SR runs ───────────────────────────────────────────────────────
all_runs = []
for session in sessions_available:
    session_path = os.path.join(NB_DIR, session)
    sr_roots = sorted([
        d for d in os.listdir(session_path)
        if d.startswith('sr_output') and os.path.isdir(os.path.join(session_path, d))
    ])
    for sr_root_name in sr_roots:
        if sr_root_name == 'sr_output':
            psf_type = 'measured'
        elif sr_root_name.startswith('sr_output_gaussian'):
            psf_type = sr_root_name.replace('sr_output_', '')
        else:
            psf_type = sr_root_name

        sr_root = os.path.join(session_path, sr_root_name)
        for calib_name in sorted(os.listdir(sr_root)):
            calib_path = os.path.join(sr_root, calib_name)
            if not os.path.isdir(calib_path):
                continue
            for rep_name in sorted(os.listdir(calib_path)):
                rep_path = os.path.join(calib_path, rep_name)
                if not os.path.isdir(rep_path) or not rep_name.startswith('rep'):
                    continue
                if os.path.exists(os.path.join(rep_path, 'done.flag')):
                    all_runs.append({
                        'session':  session,
                        'calib':    calib_name,
                        'rep':      rep_name,
                        'sr_dir':   rep_path,
                        'psf_type': psf_type,
                    })

psf_types = sorted(set(r['psf_type'] for r in all_runs))
print(f"\nPSF types found: {psf_types}")
print(f"{len(all_runs)} SR run(s) ready:")
for r in all_runs[:5]:
    print(f"  {r['session']} / {r['calib']} / {r['rep']} [{r['psf_type']}]")
if len(all_runs) > 5:
    print(f"  ... and {len(all_runs) - 5} more")

## Short labels

In [ ]:
def short_calib_label(calib_name):
    if 'nominal' in calib_name:
        return 'nominal'
    parts = calib_name.split('_')
    date_part = parts[2][4:]
    time_part = parts[3][:4]
    return f'cal_{date_part}_{time_part}'

def run_label(run):
    return f"{short_calib_label(run['calib'])} / {run['rep']}"

for r in all_runs[:3]:
    print(f"  {run_label(r)}  [{r['psf_type']}]")

## ROI Definitions

- `roi` — `(row_start, row_end, col_start, col_end)` in **HR pixel coordinates** (1536 x 2048)
- Use the viewer cell below (1/4 scale, multiply coords by 4)

In [ ]:
SESSION_ROIS = {
    "MONO_BARCODES_1PIXELSHIFT_20260407_141528": [
        {"label": "2 mil", "roi": (None, None, None, None), "pitch_mil": 2},
        {"label": "3 mil", "roi": (None, None, None, None), "pitch_mil": 3},
        {"label": "5 mil", "roi": (None, None, None, None), "pitch_mil": 5},
    ],
}

for session in sessions_available:
    assert session in SESSION_ROIS, f"Missing SESSION_ROIS entry for: {session}"
    for bc in SESSION_ROIS[session]:
        assert None not in bc["roi"],       f"Fill in ROI for {session} / {bc['label']}"
        assert bc["pitch_mil"] is not None, f"Fill in pitch_mil for {session} / {bc['label']}"

print("All ROIs defined:")
for session, barcodes in SESSION_ROIS.items():
    if session in sessions_available:
        print(f"  {session}  ->  {', '.join(str(bc['pitch_mil'])+' mil' for bc in barcodes)}")

## Image Viewer (1/4 scale)

In [ ]:
DS = 4

n_sessions = len(sessions_available)
fig, axes  = plt.subplots(1, n_sessions, figsize=(6 * n_sessions, 5), squeeze=False)

for ax, session in zip(axes[0], sessions_available):
    run = next(r for r in all_runs if r['session'] == session)
    img = np.array(Image.open(os.path.join(run['sr_dir'], 'native_2x.png')))
    ax.imshow(img[::DS, ::DS], cmap='gray', interpolation='nearest')
    ax.set_title(session, fontsize=7)
    ax.set_xlabel(f'col//{DS}  (x {DS} -> HR px)')
    ax.set_ylabel(f'row//{DS}  (x {DS} -> HR px)')

plt.suptitle(f'Native-2x at 1/{DS} scale', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
ROI_COLORS = ["red", "lime", "cyan"]

n_sessions = len(sessions_available)
fig, axes = plt.subplots(1, n_sessions, figsize=(6 * n_sessions, 5), squeeze=False)

for ax, session in zip(axes[0], sessions_available):
    run = next(r for r in all_runs if r['session'] == session)
    img = np.array(Image.open(os.path.join(run['sr_dir'], 'native_2x.png')))
    ax.imshow(img[::DS, ::DS], cmap='gray', interpolation='nearest')

    for i, bc in enumerate(SESSION_ROIS[session]):
        r0, r1, c0, c1 = bc["roi"]
        rect = patches.Rectangle(
            (c0 / DS, r0 / DS), (c1 - c0) / DS, (r1 - r0) / DS,
            linewidth=1.5, edgecolor=ROI_COLORS[i % len(ROI_COLORS)],
            facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(c0 / DS, r0 / DS - 3, bc["label"],
                color=ROI_COLORS[i % len(ROI_COLORS)], fontsize=7, va='bottom')

    ax.set_title(session, fontsize=7)

plt.suptitle('ROI overlay (Native-2x)', fontsize=10)
plt.tight_layout()
plt.show()

## Decode Function

In [ ]:
N_TRIALS   = 25
MAX_JITTER = 2
RNG        = np.random.default_rng(42)


def decode_confidence(img_gray, roi, n_trials=N_TRIALS, max_jitter=MAX_JITTER):
    r0, r1, c0, c1 = roi
    H, W = img_gray.shape

    centre_res = zxingcpp.read_barcodes(img_gray[r0:r1, c0:c1])
    text       = centre_res[0].text if centre_res else None

    successes = 0
    for _ in range(n_trials):
        dr = int(RNG.integers(-max_jitter, max_jitter + 1))
        dc = int(RNG.integers(-max_jitter, max_jitter + 1))
        rr0 = max(0, r0 + dr);  rr1 = min(H, r1 + dr)
        rc0 = max(0, c0 + dc);  rc1 = min(W, c1 + dc)
        crop = img_gray[rr0:rr1, rc0:rc1]
        if crop.size > 0 and zxingcpp.read_barcodes(crop):
            successes += 1

    return text, successes / n_trials


def crop_roi(img_gray, roi):
    r0, r1, c0, c1 = roi
    return img_gray[r0:r1, c0:c1]


print(f"decode_confidence ready  (N_TRIALS={N_TRIALS}, max_jitter={MAX_JITTER} px)")

## Run Decoding

In [ ]:
all_results = []

for run in all_runs:
    session  = run['session']
    sr_dir   = run['sr_dir']
    barcodes = SESSION_ROIS[session]
    imgs     = {m: np.array(Image.open(os.path.join(sr_dir, f'{name_map[m]}.png')))
                for m in METHOD_NAMES}

    print(f"\n{'='*60}")
    print(f"  {run_label(run)} [{run['psf_type']}]")
    print(f"{'='*60}")

    for method in METHOD_NAMES:
        print(f"\n  [{method}]")
        for bc in barcodes:
            text, conf = decode_confidence(imgs[method], bc["roi"])
            all_results.append({
                "session":    session,
                "calib":      run['calib'],
                "rep":        run['rep'],
                "run_label":  run_label(run),
                "psf_type":   run['psf_type'],
                "method":     method,
                "label":      bc["label"],
                "pitch_mil":  bc["pitch_mil"],
                "text":       text,
                "confidence": conf,
            })
            status = f"'{text}'" if text else "FAIL"
            print(f"    {bc['label']:8s}  {bc['pitch_mil']} mil  "
                  f"conf={conf:.2f}  -> {status}")

## Visual Comparison: Cropped Barcodes

In [ ]:
for run in all_runs:
    session  = run['session']
    sr_dir   = run['sr_dir']
    barcodes = SESSION_ROIS[session]
    imgs     = {m: np.array(Image.open(os.path.join(sr_dir, f'{name_map[m]}.png')))
                for m in METHOD_NAMES}
    run_res  = [r for r in all_results
                if r['calib'] == run['calib'] and r['rep'] == run['rep']
                and r['session'] == session and r['psf_type'] == run['psf_type']]

    n_bc, n_met = len(barcodes), len(METHOD_NAMES)
    fig, axes = plt.subplots(n_bc, n_met, figsize=(5 * n_met, 3 * n_bc), squeeze=False)

    for row, bc in enumerate(barcodes):
        for col, method in enumerate(METHOD_NAMES):
            ax   = axes[row][col]
            ax.imshow(crop_roi(imgs[method], bc["roi"]), cmap="gray", interpolation="nearest")
            res  = next(r for r in run_res
                        if r["method"] == method and r["pitch_mil"] == bc["pitch_mil"])
            ax.set_title(f"{method}\nconf={res['confidence']:.2f}  '{res['text'] or chr(8212)}'",
                         fontsize=8)
            ax.axis("off")
        axes[row][0].set_ylabel(f"{bc['label']}\n{bc['pitch_mil']} mil",
                                fontsize=9, rotation=0, ha="right", va="center", labelpad=5)

    plt.suptitle(f"{run_label(run)} [{run['psf_type']}]", fontsize=9, y=1.01)
    plt.tight_layout()
    plt.show()

## Decode Rate Plot (2-10 mil, mono_decoding.png style)

In [ ]:
PLOT_COLORS = {"Native-2x": "cornflowerblue", "SAA": "tab:green", "SAA+IBP": "tab:orange"}
PLOT_LABELS = {"Native-2x": "Raw (single image)", "SAA": "SAA", "SAA+IBP": "IBP"}
PLOT_MARKERS = {"Native-2x": "o", "SAA": "o", "SAA+IBP": "o"}
markers_all = PLOT_MARKERS

fig, ax = plt.subplots(figsize=(10, 6))

for method in METHOD_NAMES:
    buckets = defaultdict(list)
    for r in all_results:
        if r['method'] == method:
            buckets[r['pitch_mil']].append(r['confidence'])

    # Extend to 2-10 mil, fill missing with 100%
    extended = {}
    for p in range(2, 11):
        if p in buckets:
            extended[p] = np.mean(buckets[p])
        else:
            extended[p] = 1.0

    pitches = sorted(extended)
    confs   = [extended[p] * 100 for p in pitches]
    ax.plot(pitches, confs,
            color=PLOT_COLORS[method], marker=PLOT_MARKERS[method],
            markersize=6, linewidth=1.5, label=PLOT_LABELS[method])

    # Individual rep results for measured pitches
    for r in all_results:
        if r['method'] == method and r['pitch_mil'] in buckets:
            ax.scatter(r['pitch_mil'], r['confidence'] * 100,
                       color=PLOT_COLORS[method], alpha=0.25, s=25, zorder=1)

ax.set_xticks(range(2, 11))
ax.set_xlabel("Barcode Pitch (mil)", fontsize=12)
ax.set_ylabel("Decode Rate (%)", fontsize=12)
ax.set_title("Barcode Decode Rate by Pitch and Method (Optimal Focus, 1.0px Shift, Mono Camera Downsampled)",
             fontsize=12)
ax.tick_params(labelsize=10)
ax.set_ylim(-5, 105)
ax.set_xlim(1.5, 10.5)
ax.yaxis.set_major_locator(plt.MultipleLocator(20))
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc="upper left")

plt.tight_layout()
plt.show()

## Per-Calibration Breakdown

In [ ]:
calib_names = sorted(set(r['calib'] for r in all_results))

for calib in calib_names:
    calib_res = [r for r in all_results if r['calib'] == calib]

    fig, ax = plt.subplots(figsize=(10, 6))

    for method in METHOD_NAMES:
        buckets = defaultdict(list)
        for r in calib_res:
            if r['method'] == method:
                buckets[r['pitch_mil']].append(r['confidence'])

        extended = {}
        for p in range(2, 11):
            if p in buckets:
                extended[p] = np.mean(buckets[p])
            else:
                extended[p] = 1.0

        pitches = sorted(extended)
        confs   = [extended[p] * 100 for p in pitches]
        ax.plot(pitches, confs,
                color=PLOT_COLORS[method], marker=PLOT_MARKERS[method],
                markersize=6, linewidth=1.5, label=PLOT_LABELS[method])

    ax.set_xticks(range(2, 11))
    ax.set_xlabel("Barcode Pitch (mil)", fontsize=12)
    ax.set_ylabel("Decode Rate (%)", fontsize=12)
    ax.set_title(f"Barcode Decode Rate (1.0px Shift, Mono Camera Downsampled)\n{short_calib_label(calib)}",
                 fontsize=12)
    ax.tick_params(labelsize=10)
    ax.set_ylim(-5, 105)
    ax.set_xlim(1.5, 10.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(20))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10, loc="upper left")

    plt.tight_layout()
    plt.show()